# 집단 탐색 — 어느 유저 집단부터 검증해야 하나

`Modeling.ipynb` 까지로 **행마다 편차**(실제 − 기대 p)가 준비됐음. 이 노트북은 그 편차를 써서
**"먼저 검증할 가설 1~K위"** 를 뽑음. 구현 4번에 해당함.

| 단계 | 내용 |
|---|---|
| **1** | 준비 |
| **2** | 데이터 불러오기 (`mart` 탐색 축 + `pred` 편차) · 점검 |
| **3** | 조건 재료 만들기 — 연속 변수는 **정상 기간 분위수 5구간**, 범주 변수는 값별 (드문 값은 "기타") |
| **4** | **점수 함수** — 기준선 보정을 포함한 집단 점수 |
| **5** | **빔 서치** (빔 폭 20, 조건 최대 3개, 최소 100명) |
| **6** | 결과 — 중복 제거 · **무주입 영분포와 대조** → 상위 K개 |

#### 점수 함수 (`Modeling.ipynb` 6절에서 확정)

```
집단 S 의 점수 = 평가 편차 − 기준선

  평가 행 1만 이상 → 기준선 = 학습 편차 + 기울기 × (평가 평균날짜 − 학습 평균날짜)
  평가 행 1만 미만 → 기준선 = 전체 기준선 (+1.46%p)
```

집단마다 합 7개로 계산됨 — 평가 `n · Σdev` + 정상 `n · Σd · Σdev · Σd² · Σd·dev`.
⚠️ **보정은 탐색 중에** 들어가야 함. 사후 적용하면 1단계에서 잘린 조건이 조합으로도 못 올라옴.

#### 이 노트북이 쓰는 것

| 파일 | 무엇 |
|---|---|
| `data/mart.parquet` | 탐색 축이 될 변수들 |
| `data/pred.parquet` | 행마다 `day`, `dev` (= 실제 − p) |

**모델은 돌리지 않음.** 예측은 `Modeling.ipynb` 4절에서 이미 끝나 파일에 저장돼 있음.

## 1. 준비

In [2]:
# 집단 탐색에 쓰는 라이브러리임 (impok 환경)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)
print('pandas {} / numpy {}'.format(pd.__version__, np.__version__))

pandas 2.3.2 / numpy 1.26.4


## 2. 데이터 불러오기 · 점검

In [3]:
_t0 = time.time()

# ── 1) 탐색 축이 될 변수만 읽음 (모델 입력과 꼭 같을 필요는 없음) ────────
AX_A = ['days_since_signup', 'base_dow']                                   # 시간 구조
AX_B = ['city', 'age', 'gender', 'channel', 'profile']                     # 유저 특성
AX_C = ['has_tx', 'payment_method_id', 'payment_plan_days', 'plan_list_price',
        'actual_amount_paid', 'discount', 'is_auto_renew', 'is_cancel', 'days_to_expire']
AX_D = ['act_days', 'mins_per_day', 'pct_full', 'pct_unique',
        'days_since_last', 'trend', 'weekend_ratio', 'no_habit']           # 평소 습관
AXES = AX_A + AX_B + AX_C + AX_D

mart = pd.read_parquet('data/mart.parquet', columns=['u', 'day'] + AXES)
pred = pd.read_parquet('data/pred.parquet', columns=['u', 'day', 'd1', 'p', 'dev'])
print('mart {:,}행 x {}열 / pred {:,}행   ({:.0f}초)'.format(
    len(mart), mart.shape[1], len(pred), time.time() - _t0))

# ── 2) 두 파일이 같은 행 순서인지 (merge 하지 않고 쓰므로 필수 점검) ─────
assert len(mart) == len(pred), '행 수가 다름'
assert (mart['u'].to_numpy()   == pred['u'].to_numpy()).all(),   'u 가 어긋남'
assert (mart['day'].to_numpy() == pred['day'].to_numpy()).all(), 'day 가 어긋남'
print('행 정렬 일치 (u, day)')

# ── 3) 기간 분할 — Modeling.ipynb 과 같은 기준이어야 함 ─────────────────
TRAIN_LAST = 14                                  # 학습 기간 마지막 기준일 = 3/15
is_train = (mart['day'] <= TRAIN_LAST).to_numpy()
is_eval  = ~is_train
day = mart['day'].to_numpy().astype('float64')   # 0 = 3/1 ... 29 = 3/30
dev = pred['dev'].to_numpy().astype('float64')   # 편차 = 실제 − p
print('학습 {:,}행 / 평가 {:,}행'.format(is_train.sum(), is_eval.sum()))

# ── 4) Modeling.ipynb 4절 검증값과 대조 ─────────────────────────────────
for 이름, m_, 기대 in [('학습 편차', is_train, -0.00), ('평가 편차', is_eval, 1.25)]:
    실제 = dev[m_].mean() * 100
    print('  {} {:+.4f}%p   (기대 {:+.2f})'.format(이름, 실제, 기대))
    assert abs(실제 - 기대) < 0.05, '{} 가 Modeling.ipynb 과 다름'.format(이름)

# ── 5) 3/30 은 평가 기간에서 뺌 (10/8 확정) ─────────────────────────────
# 결과일 3/31 이 데이터 마지막 날이라 DAU 가 같은 요일보다 −3.9% 라 D1 의 분자가 깎임.
# 3/2~3/15 편차로 그은 직선에서 3/30 은 −4.12%p(20.4σ) 벗어남 (반대쪽 경계 3/1 은 0.3σ 로 정상).
# 빼면 평가가 14일이 되어 **요일이 정확히 2번씩** 들어감 (전에는 목요일만 3번이었음).
# 일반 규칙: 데이터가 끝나는 날은 결과일로 쓸 수 없으므로 그 전날을 기준일에서 뺌.
EXCL_LAST = True                                 # False 로 두면 민감도 분석용 비교가 됨
eval_m = is_eval & (day <= 28) if EXCL_LAST else is_eval
print('\n평가 구간 {:,}행 (3/30 {})   전체 기준선 {:+.3f}%p'.format(
    eval_m.sum(), '제외' if EXCL_LAST else '포함', dev[eval_m].mean() * 100))
print('전체 {:.0f}초'.format(time.time() - _t0))

mart 17,818,632행 x 26열 / pred 17,818,632행   (2초)
행 정렬 일치 (u, day)
학습 8,802,984행 / 평가 9,015,648행
  학습 편차 -0.0000%p   (기대 -0.00)
  평가 편차 +1.2467%p   (기대 +1.25)

평가 구간 8,407,908행 (3/30 제외)   전체 기준선 +1.462%p
전체 3초


## 3. 조건 재료 만들기

빔 서치가 조합할 **기본 조건을 분석 전에 전부 고정**함 (설계 3절). 데이터를 보고 조건을 바꾸면
다중검정 부담을 셀 수 없게 됨.

| 변수 종류 | 만드는 조건 |
|---|---|
| 연속 | 학습(정상) 기간 **분위수 5구간** → 구간 조건 + 누적 조건(`≤`, `>`) |
| 범주 | 값별 `=` 조건 + `≠` 조건. 행 1,500 미만(≒ 유저 100명)인 값은 **"기타"** 로 묶음 |
| 0/1 | `예` / `아니오` |

**결측은 버리지 않고 "(결측)" 이라는 하나의 값**으로 둠 — 빔 서치는 NaN 조건을 만들 수 없음.

#### 분위수를 안 쓰는 축 둘

분포가 한쪽에 뭉쳐 분위수가 듣지 않아, **『EDA 정리』와 `Modeling.ipynb` 6절에서 쓰던 구간**을 그대로 씀.

| 축 | 분위수로 자르면 | 그래서 |
|---|---|---|
| `act_days` | `≤12` 가 한 칸이 되어 **기울기가 가장 가파른 1~3일 집단이 묻힘** | 0(미측정)/1-3/4-10/11-20/21-28 |
| `days_since_last` | 84%가 2/28까지 들어서 **2칸으로 주저앉음** | 0일/1-2/3-6/7-13/14일+ |

6절 표와 바로 대조할 수 있다는 이점도 있음.

#### ⚠️ 탐색 축에서 뺀 것

| 축 | 이유 |
|---|---|
| `base_dow` (기준일 요일) | ① 기대 모델이 이미 흡수해 편차에 요일이 거의 안 남음 ② 학습 기간에 그 요일이 **2~3번뿐이라 기준선 보정 불가**(6-3 B) ③ "금요일 유저를 검증하라" 는 실무 가설이 못 됨 |
| `days_since_first_mar` | 달력 날짜의 대리라 축으로도 부적절 (`Modeling.ipynb` 5절에서 기각) |

`days_since_signup` 은 **남겨 둠** — APC 식별 문제로 해석에 주의가 필요하지만, "신규 유저"는
실무에서 가장 흔한 가설 축이고 학습 기간 15일에 모두 나타나 보정도 가능함.

In [4]:
_t0 = time.time()

# ── 조건 재료를 분석 전에 전부 고정함 (설계 3절) ────────────────────────
#   연속 변수 → 학습(정상) 기간 분위수 5구간. 구간 조건 + 누적 조건(≤, >)
#   범주 변수 → 값별 '=' 조건 + '≠' 조건
#   결측은 버리지 않고 **"(결측)" 이라는 하나의 값**으로 둠 — 빔 서치는 NaN 조건을 못 만듦
#   드문 값(행 1,500 미만 ≒ 유저 100명)은 "기타" 로 묶음
#
# ⚠️ 탐색 축에서 뺀 것
#   base_dow (기준일 요일) — ① 기대 모델이 이미 흡수해 편차에 요일이 거의 안 남고
#     ② 학습 기간에 그 요일이 2~3번뿐이라 기준선 보정이 불가능하며 (6-3 B)
#     ③ "금요일 유저를 검증하라" 는 실무 가설이 되지 못함
#   days_since_first_mar — 달력 날짜의 대리라 축으로도 부적절 (Modeling.ipynb 5절에서 기각)

MIN_ROWS = 1500        # 이보다 적게 나오는 범주 값은 "기타" 로 묶음
NQ       = 5           # 연속 변수 분위 구간 수 (3·10 구간은 민감도 분석)

# 분포가 한쪽에 뭉쳐 분위수가 안 듣는 두 축만 EDA·6절에서 쓰던 구간을 그대로 씀.
#   act_days        — 분위수로 자르면 ≤12 가 한 칸이 되어 **기울기가 가장 가파른 1~3일 집단이 묻힘**
#   days_since_last — 84%가 2/28 까지 들어서 분위수가 2칸으로 주저앉음
# 둘 다 『EDA 정리』와 Modeling.ipynb 6절에서 쓴 구간이라 결과를 바로 대조할 수 있음
CUSTOM_BINS = {'act_days'       : ([-1, 0, 3, 10, 20, 28], ['0 (미측정)', '1-3일', '4-10일', '11-20일', '21-28일']),
               'days_since_last': ([-1, 0, 2, 6, 13, 27],  ['0일(2/28까지)', '1-2일', '3-6일', '7-13일', '14일+'])}

AX_CONT = ['days_since_signup', 'payment_plan_days', 'plan_list_price', 'actual_amount_paid',
           'discount', 'days_to_expire', 'act_days', 'mins_per_day', 'pct_full',
           'pct_unique', 'days_since_last', 'trend', 'weekend_ratio']
AX_CAT  = ['city', 'age', 'gender', 'channel', 'profile', 'payment_method_id']
AX_BOOL = ['has_tx', 'is_auto_renew', 'is_cancel', 'no_habit']

AX, EDGES = {}, {}     # 축이름 → (코드 int16, 값 이름 목록).  코드 0 = (결측)


def _compact(code, lab):
    """안 쓰이는 코드를 없애고 0(결측)은 늘 유지함"""
    used = np.unique(code)
    new  = np.full(len(lab), -1, dtype='int16')
    new[used] = np.arange(len(used), dtype='int16')
    return new[code], [lab[i] for i in used]


# ── 연속 변수 — 경계는 학습 기간 분위수에서 잡아 전체에 적용함 ───────────
for c in AX_CONT:
    x = mart[c].to_numpy().astype('float64')
    if c in CUSTOM_BINS:
        e, lab = CUSTOM_BINS[c]
        e = np.array(e, dtype='float64'); e[0], e[-1] = -np.inf, np.inf
    else:
        e = np.unique(np.nanquantile(x[is_train], np.linspace(0, 1, NQ + 1)))
        e[0], e[-1] = -np.inf, np.inf
        lab = []
        for i in range(len(e) - 1):
            lo, hi = e[i], e[i + 1]
            lab.append('≤{:g}'.format(hi) if i == 0 else
                       '>{:g}'.format(lo) if i == len(e) - 2 else '{:g}~{:g}'.format(lo, hi))
    b = pd.cut(x, e)
    AX[c], EDGES[c] = _compact((b.codes + 1).astype('int16'), ['(결측)'] + lab), e

# ── 범주 변수 — 드문 값은 "기타" 로 묶음 ────────────────────────────────
for c in AX_CAT:
    s = mart[c] if isinstance(mart[c].dtype, pd.CategoricalDtype) else mart[c].astype('category')
    code = (s.cat.codes.to_numpy().astype('int32') + 1)
    lab  = ['(결측)'] + [str(v) for v in s.cat.categories]
    cnt  = np.bincount(code, minlength=len(lab))
    rare = [i for i in range(1, len(lab)) if cnt[i] < MIN_ROWS]
    if rare:
        code = np.where(np.isin(code, rare), len(lab), code)
        lab  = lab + ['기타({}개 값)'.format(len(rare))]
    AX[c] = _compact(code.astype('int16'), lab)

# ── 0/1 변수 ────────────────────────────────────────────────────────────
for c in AX_BOOL:
    v = mart[c].to_numpy()
    code = (v.astype('int16') + 1) if v.dtype == bool else \
           np.where(np.isnan(v.astype('float64')), 0, np.where(v > 0.5, 2, 1)).astype('int16')
    AX[c] = _compact(code, ['(결측)', '아니오', '예'])

# ── 조건 목록 만들기 ────────────────────────────────────────────────────
CONDS = []                                     # {이름, 축, sel(코드별 True/False)}


def add(ax, name, keep):
    K = len(AX[ax][1])
    sel = np.zeros(K, dtype=bool)
    sel[list(keep)] = True
    CONDS.append({'이름': name, '축': ax, 'sel': sel})


for ax in AX_CONT:
    code, lab = AX[ax]
    K, hasna = len(lab), (lab[0] == '(결측)')
    bins = [i for i in range(K) if lab[i] != '(결측)']
    for i in bins:                                                  # 구간
        add(ax, '{} {}'.format(ax, lab[i]), [i])
    for i in bins[:-1]:                                             # 누적 ≤
        add(ax, '{} ≤{:g}'.format(ax, EDGES[ax][bins.index(i) + 1]), bins[:bins.index(i) + 1])
    for i in bins[:-1]:                                             # 누적 >
        add(ax, '{} >{:g}'.format(ax, EDGES[ax][bins.index(i) + 1]), bins[bins.index(i) + 1:])
    if hasna:
        add(ax, '{} 결측'.format(ax), [0])

for ax in AX_CAT + AX_BOOL:
    code, lab = AX[ax]
    K = len(lab)
    for i in range(K):
        add(ax, '{} = {}'.format(ax, lab[i]), [i])
        if K > 2:                                                   # 값이 2개면 ≠ 는 = 와 같음
            add(ax, '{} ≠ {}'.format(ax, lab[i]), [j for j in range(K) if j != i])

print('축 {}개 (연속 {} · 범주 {} · 0/1 {})   값 {}개   **조건 {}개**   {:.0f}초'.format(
    len(AX), len(AX_CONT), len(AX_CAT), len(AX_BOOL),
    sum(len(v[1]) for v in AX.values()), len(CONDS), time.time() - _t0))
print('조건 3개 조합 (순서 무시, 같은 축 중복 허용 시) 약 {:,.0f}개 — 빔 서치는 이 중 '
      '약 {:,}개만 평가함 (폭 20 x 3단계)'.format(
          len(CONDS) ** 3 / 6, len(CONDS) + 2 * 20 * len(CONDS)))

# ── 점검 — 축마다 값과 행 수를 보고 해석 가능한지 눈으로 확인함 ──────────
print('\n축별 값 (행 수, 평가 기간 기준):')
for ax, (code, lab) in AX.items():
    cnt = np.bincount(code[eval_m], minlength=len(lab))
    top = ' / '.join('{} {:,}'.format(lab[i], cnt[i]) for i in np.argsort(-cnt)[:4])
    print('  {:<20s} 값 {:>2}개   {}{}'.format(ax, len(lab), top, ' / …' if len(lab) > 4 else ''))

축 23개 (연속 13 · 범주 6 · 0/1 4)   값 137개   **조건 290개**   16초
조건 3개 조합 (순서 무시, 같은 축 중복 허용 시) 약 4,064,833개 — 빔 서치는 이 중 약 11,890개만 평가함 (폭 20 x 3단계)

축별 값 (행 수, 평가 기간 기준):
  days_since_signup    값  6개   1343~2126 1,702,035 / 358~746 1,698,108 / >2126 1,693,779 / 746~1343 1,671,486 / …
  payment_plan_days    값  4개   ≤30 6,569,092 / >90 1,572,493 / 30~90 152,424 / (결측) 113,899
  plan_list_price      값  5개   99~149 3,805,486 / ≤99 1,862,490 / >300 1,633,019 / 149~300 993,014 / …
  actual_amount_paid   값  5개   99~149 3,790,974 / ≤99 1,877,722 / >300 1,633,861 / 149~300 991,452 / …
  discount             값  3개   ≤0 8,275,791 / (결측) 113,899 / >0 18,218
  days_to_expire       값  6개   9~18 1,987,765 / ≤9 1,796,760 / 18~26 1,678,680 / >47 1,527,242 / …
  act_days             값  5개   21-28일 4,396,095 / 11-20일 2,468,169 / 4-10일 909,004 / 0 (미측정) 394,280 / …
  mins_per_day         값  6개   ≤56.1611 1,655,968 / 56.1611~82.2579 1,611,145 / 82.2579~114.517 1,595,056 / 114.517~176.145 1,585,282 / …
  pct_full

## 4. 점수 함수

후보 집단 하나를 **합 12개**로 채점함. 빔 서치는 이 점수로만 순위를 매김.

```
                   평가 행 1만 이상 → 학습 편차 + 기울기 × 갭   (그 집단의 선)
기준선(S) =
                   평가 행 1만 미만 → 전체 기준선 (+1.46%p)

하락(S) = 기준선 − 평가 편차                      (양수 = 기대보다 덜 돌아옴)
t(S)   = 하락 ÷ √( 평가 편차의 분산 + 기준선의 분산 )
                                    └──── ★ 이게 핵심 ────┘
```

#### ★ 기준선도 추정값이라 오차가 있음

`기준선 = a + b·x₀` 는 회귀선의 **예측값**이므로 분산이 있음.

```
Var(기준선) = s²_날짜 × [ 1/n + 갭² / Σ(d − d̄)² ]
```

`s²_날짜` 는 **날짜별 평균 편차가 회귀선에서 벗어난 정도**임 (`Modeling.ipynb` 6-3 에서
합성 데이터 + 블록 부트스트랩으로 참값의 0.90배임을 확인한 방식). 같은 날의 행은 함께
흔들리므로 행을 독립으로 세면 안 됨.

이걸 빼먹으면 **기준선을 정확히 안다고 가정**하는 셈이라 t 가 크게 부풀고, 하락이 없는
데이터에서도 상위 가설이 쏟아짐. 넣고 빼고의 차이:

| | 무시 | **넣음** |
|---|---|---|
| 최고 t | 16.49 | **7.47** |
| t > 4 인 후보 | 4,436개 | **468개** |
| 1위 가설 | 15만 명 × 2.05%p (보정 잔차가 큰 큰 집단) | 1.6만 명 × 6.60%p |
| 습관 집단 t (미측정/4-10일) | 6.2 / 7.3 | **1.0 / 1.2** |

→ **"큰 집단의 작은 잔차" 가 ±기준선에 묻혀 더는 올라오지 못함.**
기울기를 잘 못 재는 집단이 자동으로 불리해지므로, 게이트·축소로 억지로 하려던 일이
표준오차 하나로 해결됨 (6-5 에서 둘 다 기각됐던 이유이기도 함).

#### ⚠️ 유효 인원 — 한 유저가 평가 기간에 최대 14행

행으로 세면 유의성이 √14배쯤 부풀고, 더 나쁜 것은 **가볍게 듣는 유저일수록 행이 적어 불리해진다**는
점임(하필 하락이 가장 잘 일어나는 쪽임). 그래서 **유저별 행 수의 역수**를 가중치로 두고 그 합을
인원으로 씀. 전체에 적용하면 992,683명이 나옴.

#### 빠르게 세는 법

부모 집단의 행에서 **축마다 "코드별 합"** 을 `bincount` 로 한 번에 구해 두면, 그 축의 조건은
**고른 코드의 합을 더하기만** 하면 됨. 조건 하나마다 1,700만 행을 다시 훑지 않아도 됨.
`(코드 × 날짜)` 합 2개가 추가로 필요한데(위 `s²_날짜`), 비용은 20%만 늘어남.

In [5]:
_t0 = time.time()

# ── 1) 행을 [학습 | 평가] 순으로 재배치하고 3/30 은 버림 ────────────────
_seg  = np.where(is_train, 0, np.where(day <= 28, 1, 2))
_keep = np.argsort(_seg, kind='stable')[: (_seg < 2).sum()]
N_TR, N = int(is_train.sum()), int((_seg < 2).sum())
DAY  = day[_keep]
DEV  = dev[_keep]
CODE = {ax: AX[ax][0][_keep] for ax in AX}
LAB  = {ax: AX[ax][1] for ax in AX}
DAYI = DAY[:N_TR].astype('int32')                  # 학습 기간 날짜 번호 0~14
NDAY = int(DAYI.max()) + 1
DAYS = np.arange(NDAY, dtype='float64')

# ── 2) 유효 인원 가중치 — 한 유저가 평가 기간에 최대 14행 ────────────────
_u   = mart['u'].to_numpy()[_keep]
_cnt = np.bincount(_u[N_TR:], minlength=_u.max() + 1).astype('float64')
WU   = np.zeros(N); WU[N_TR:] = 1.0 / _cnt[_u[N_TR:]]

MIN_USERS, MIN_FIT = 100, 10000
BASE_ALL = DEV[N_TR:].mean()
_WT = [None, DAY, DEV, DAY * DAY, DAY * DEV]
_WE = [None, WU,  DEV, DEV * DEV, DAY]
print('작업 배열 {:,}행 (학습 {:,} + 평가 {:,})   유효 인원 {:,.0f}명   전체 기준선 {:+.3f}%p'.format(
    N, N_TR, N - N_TR, WU.sum(), BASE_ALL * 100))


def cell_sums(idx):
    """축마다 코드별 합 10개 + **(코드 × 날짜)별 합 2개**를 구함.
       뒤의 둘은 기준선 추정의 불확실성(날짜 수준)을 재는 데 씀"""
    k = np.searchsorted(idx, N_TR)
    it, ie = idx[:k], idx[k:]
    wt = [np.ones(len(it))] + [w[it] for w in _WT[1:]]
    we = [np.ones(len(ie))] + [w[ie] for w in _WE[1:]]
    di, vt = DAYI[it], DEV[it]
    out = {}
    for ax in AX:
        K = len(LAB[ax])
        ct, ce = CODE[ax][it], CODE[ax][ie]
        ci = ct * NDAY + di
        out[ax] = (np.stack([np.bincount(ct, weights=w, minlength=K) for w in wt]),
                   np.stack([np.bincount(ce, weights=w, minlength=K) for w in we]),
                   np.bincount(ci, minlength=K * NDAY).reshape(K, NDAY).astype('float64'),
                   np.bincount(ci, weights=vt, minlength=K * NDAY).reshape(K, NDAY))
    return out


def score(T, E, Gn, Gv, raw_t=False):
    """T=[n,Σd,Σdev,Σd²,Σd·dev](학습)  E=[n,Σw,Σdev,Σdev²,Σd](평가)
       Gn,Gv = 날짜별 행 수·Σdev (학습).  반환: 인원, 하락(%p), t, ±기준선(%p), 기울기(%p/일)"""
    n, Sd, Sv, Sdd, Sdv = T
    ne, Su, Sve, Svve, Sde = E
    if ne < 1 or n < 2 or Su < MIN_USERS:
        return None
    mu_ev = Sve / ne
    base, b, var_base = BASE_ALL, 0.0, 0.0
    if ne >= MIN_FIT:
        Sxx = Sdd - Sd * Sd / n
        ok  = Gn > 0
        if Sxx > 0 and ok.sum() >= 3:
            dbar = Sd / n
            b    = (Sdv - Sd * Sv / n) / Sxx
            a    = Sv / n - b * dbar
            gap  = Sde / ne - dbar
            base = a + b * (Sde / ne)
            # 날짜별 평균 편차가 회귀선에서 벗어난 정도로 s² 을 잼 (6-3 에서 검증한 방식)
            md = Gv[ok] / Gn[ok]
            e  = md - (a + b * DAYS[ok])
            s2 = (Gn[ok] * e * e).sum() / max(ok.sum() - 2, 1)
            var_base = s2 * (1.0 / n + gap * gap / Sxx)     # 회귀선 예측값의 분산
    drop = base - mu_ev
    var  = max(Svve / ne - mu_ev * mu_ev, 1e-12)
    se2  = var / Su + (0.0 if raw_t else var_base)          # ★ 기준선의 오차까지 더함
    t    = drop / np.sqrt(se2)
    return Su, drop * 100, t, np.sqrt(var_base) * 100, b * 100


print('최소 인원 {}명 · 보정 하한 평가 {:,}행   ({:.0f}초)'.format(MIN_USERS, MIN_FIT, time.time() - _t0))

# ── 점검: Modeling.ipynb 6-3 의 ±보정 과 맞는지 습관 집단으로 대조함 ─────
_S = cell_sums(np.arange(N))
_ax = 'act_days'
print('\n[점검] 습관 집단의 ±기준선 — Modeling.ipynb 6-3 의 ±보정 과 같아야 함')
print('{:<14s}{:>10s}{:>10s}{:>12s}{:>12s}'.format('집단', '기울기', '±기준선', '(6-3 ±보정)', 't 비교'))
_ref = {'0 (미측정)': 0.805, '1-3일': 0.656, '4-10일': 0.650, '11-20일': 0.299, '21-28일': 0.118}
for _i, _l in enumerate(LAB[_ax]):
    if _l == '(결측)':
        continue
    _sel = np.zeros(len(LAB[_ax]), bool); _sel[_i] = True
    _r  = score(_S[_ax][0][:, _sel].sum(1), _S[_ax][1][:, _sel].sum(1),
                _S[_ax][2][_sel].sum(0), _S[_ax][3][_sel].sum(0))
    _r0 = score(_S[_ax][0][:, _sel].sum(1), _S[_ax][1][:, _sel].sum(1),
                _S[_ax][2][_sel].sum(0), _S[_ax][3][_sel].sum(0), raw_t=True)
    print('{:<14s}{:>+10.3f}{:>10.3f}{:>12.3f}{:>8.1f} → {:.1f}'.format(
        _l, _r[4], _r[3], _ref.get(_l, np.nan), _r0[2], _r[2]))

작업 배열 17,210,892행 (학습 8,802,984 + 평가 8,407,908)   유효 인원 992,683명   전체 기준선 +1.462%p
최소 인원 100명 · 보정 하한 평가 10,000행   (2초)

[점검] 습관 집단의 ±기준선 — Modeling.ipynb 6-3 의 ±보정 과 같아야 함
집단                   기울기      ±기준선   (6-3 ±보정)        t 비교
0 (미측정)           +0.353     0.840       0.805     6.2 → 1.0
1-3일              +0.446     0.685       0.656    -0.1 → -0.0
4-10일             +0.354     0.678       0.650     7.3 → 1.2
11-20일            +0.169     0.312       0.299     3.4 → 0.9
21-28일            +0.005     0.123       0.118     1.5 → 0.6


## 5. 빔 서치

| 설정 | 값 | 비고 |
|---|---|---|
| 빔 폭 | 20 | 설계 3절, **분석 전에 고정** |
| 조건 최대 | 3개 | 같은 축은 두 번 안 씀 |
| 최소 인원 | 100명 | 유효 인원 기준 |
| 무의미 조건 차단 | 집단이 **5% 미만으로 줄면 버림** | `is_cancel ≠ (결측)` 처럼 98%를 그대로 남기는 조건이 t 를 소수점만 올리며 상위를 채우는 것을 막음 |
| 중복 제거 | 행 겹침 **50% 이상이면 하나만** | DSSD 식 |

```
1단계  조건 225개      → 점수 → 상위 20
2단계  20 × 조건       → 점수 → 상위 20
3단계  20 × 조건       → 점수 → 상위 20
       ↓
모은 후보에서 중복 제거 → 상위 K개
```

⚠️ **보정은 탐색 중에** 들어감. 사후에 붙이면 1단계에서 잘린 조건이 조합으로도 못 올라옴.

이 셀은 탐색을 **함수 두 개로 정의**하고 마지막에 한 번 호출한다 — 8절 주입 실험이 같은 함수를 수십 번 부르기
때문에, 로직이 한 군데에만 있어야 어긋나지 않는다.

| 함수 | 하는 일 |
|---|---|
| `빔서치(dev, 조용히=True)` | 넘겨받은 편차 배열로 처음부터 탐색. 반환은 **중복 제거 전** 후보 전부 (t 내림차순) |
| `추려내기(후보, topk)` | 행이 50% 이상 겹치는 가설을 합쳐 상위 몇 개만 남김 |

In [6]:
_t0 = time.time()

BEAM, DEPTH = 20, 3    # 빔 폭 · 조건 최대 개수 (설계 3절, 분석 전에 고정)
MIN_CUT  = 0.95        # 조건을 붙여 집단이 이보다 덜 줄면 버림 — "is_cancel ≠ (결측)" 처럼
                       # 98%를 그대로 남기는 조건이 t 를 소수점만 올리며 상위를 채우는 것을 막음
MAX_JACC = 0.5         # 보고할 때 행이 이만큼 겹치면 같은 가설로 보고 하나만 남김 (DSSD 식)
TOPK     = 10

_root  = np.arange(N)
_apply = lambda idx, c: idx[c['sel'][CODE[c['축']][idx]]]
_memo  = {(): _root}


def rows(d):
    """설명이 고르는 행 — 앞부분을 기억해 두고 조건 하나씩만 더 걸러냄"""
    k = tuple(c['이름'] for c in d)
    if k not in _memo:
        _memo[k] = _apply(rows(d[:-1]), d[-1])
    return _memo[k]


def 빔서치(dev_use, 조용히=True):
    """조건을 하나씩 붙여가며 t 가 높은 집단을 찾음 (빔 폭 20 · 조건 3개까지).
       넘겨받은 편차 배열로 전역 DEV 를 갈아끼우므로 8절 주입 실험도 이 함수를 그대로 씀.
       반환: [(점수결과, 설명), ...] 을 t 내림차순으로 — **중복 제거 전**"""
    global DEV, _WT, _WE, BASE_ALL
    t0 = time.time()
    DEV, BASE_ALL = dev_use, dev_use[N_TR:].mean()
    _WT = [None, DAY, DEV, DAY * DAY, DAY * DEV]
    _WE = [None, WU,  DEV, DEV * DEV, DAY]
    beam, allc = [([], _root)], []
    for 단계 in range(1, DEPTH + 1):
        cand = {}
        for desc, idx in beam:
            S, used = cell_sums(idx), {c['축'] for c in desc}
            for c in CONDS:
                if c['축'] in used:                         # 같은 축을 두 번 쓰지 않음
                    continue
                a = c['sel']
                T, E = S[c['축']][0][:, a].sum(1), S[c['축']][1][:, a].sum(1)
                if T[0] + E[0] > MIN_CUT * len(idx):        # 사실상 아무것도 안 거름
                    continue
                r = score(T, E, S[c['축']][2][a].sum(0), S[c['축']][3][a].sum(0))
                if r is None:
                    continue
                key = frozenset(x['이름'] for x in desc) | {c['이름']}
                if key not in cand or r[2] > cand[key][0][2]:
                    cand[key] = (r, desc + [c], idx, c)
        rank = sorted(cand.values(), key=lambda x: -x[0][2])
        allc += [(r, d) for r, d, _, _ in rank]
        beam  = [(d, _apply(p, c)) for _, d, p, c in rank[:BEAM]]
        if not 조용히:
            print('{}단계 — 후보 {:,}개 평가, 상위 {} 유지   최고 t {:.2f}   ({:.0f}초)'.format(
                단계, len(cand), len(beam), rank[0][0][2] if rank else 0, time.time() - t0))
    allc.sort(key=lambda x: -x[0][2])
    return allc


def 추려내기(allc, topk=TOPK):
    """행이 MAX_JACC 이상 겹치는 가설을 합쳐 상위 topk 만 남김 (DSSD 식)"""
    kept, sets = [], []
    for r, d in allc[:150]:
        s = rows(d)
        if any(len(np.intersect1d(s, o, True)) / len(np.union1d(s, o)) >= MAX_JACC for o in sets):
            continue
        kept.append((r, d)); sets.append(s)
        if len(kept) >= topk:
            break
    return kept


# ── 실행 (주입 없는 실제 데이터) ────────────────────────────────────────
_all  = 빔서치(DEV, 조용히=False)
_kept = 추려내기(_all)

res = pd.DataFrame([{'가설': ' AND '.join(c['이름'] for c in d), '조건': len(d), '인원': r[0],
                     '하락(%p)': r[1], '±기준선': r[3], '기울기': r[4], 't': r[2]}
                    for r, d in _kept])
print('\n★ 상위 {}개 (후보 {:,}개 중, 중복 제거 후)'.format(len(res), len(_all)))
print(res.to_string(formatters={'인원': '{:,.0f}'.format}, float_format='{:+.2f}'.format))
print('\n유의 수준별 후보 수:  t>2 {:,}개 / t>4 {:,}개 / t>6 {:,}개'.format(
    *[sum(1 for r, _ in _all if r[2] > x) for x in (2, 4, 6)]))
print('⚠️ 실제 데이터에는 하락이 없음 — 위 목록은 전부 **거짓 경보**이고, '
      '주입 실험의 영분포(기준선)로 씀 (계획서 7.2)')
print('전체 {:.0f}초'.format(time.time() - _t0))

1단계 — 후보 225개 평가, 상위 20 유지   최고 t 4.34   (14초)
2단계 — 후보 3,178개 평가, 상위 20 유지   최고 t 5.77   (32초)
3단계 — 후보 2,580개 평가, 상위 20 유지   최고 t 7.47   (35초)

★ 상위 2개 (후보 5,983개 중, 중복 제거 후)
                                                                  가설  조건     인원  하락(%p)  ±기준선   기울기     t
0              payment_plan_days 결측 AND channel ≠ 4 AND act_days ≤20   3 16,302   +6.60 +0.80 +0.36 +7.47
1  is_cancel ≠ 아니오 AND days_since_signup ≤746 AND profile = complete   3  7,548   +4.26 +0.71 +0.18 +4.86

유의 수준별 후보 수:  t>2 2,375개 / t>4 468개 / t>6 27개
⚠️ 실제 데이터에는 하락이 없음 — 위 목록은 전부 **거짓 경보**이고, 주입 실험의 영분포(기준선)로 씀 (계획서 7.2)
전체 38초


## 6. 결과 판정 (2026-10-08)

### 실행 결과 — 중복 제거 후 상위 2개

| 가설 | 인원 | 하락 | ±기준선 | 기울기 | t |
|---|---|---|---|---|---|
| `payment_plan_days 결측` AND `channel ≠ 4` AND `act_days ≤20` | 16,302 | **6.60%p** | ±0.80 | +0.36 | **7.47** |
| `is_cancel ≠ 아니오` AND `days_since_signup ≤746` AND `profile = complete` | 7,548 | 4.26%p | ±0.71 | +0.18 | 4.86 |

`payment_plan_days 결측` = **거래 기록이 없는 무료 유저**임. 모델이 가장 못 맞추는 집단이기도 함.

### ⚠️ 이 목록은 전부 **거짓 경보**임

**실제 데이터에는 하락이 없음** (평가 기간 D1이 학습보다 +0.40%p 높음). 그런데도 t 가 7.47 로 나옴.
**합격선은 BH 가 아니라 이 실측값으로 잡는다.** 후보 수만으로 이론 합격선을 내면 t ≈ 3.8 인데, 그 선으로 자르면 **하락이 하나도 없는 이 데이터에서 468개가 "유의한 하락"으로 통과한다.** 이론이 깨지는 이유는 ① 후보 6,000개가 행을 잔뜩 공유해 독립이 아니고 ② t 가 정확히 t분포를 따르지 않으며 ③ 빔 서치가 애초에 **극값만 추려서** 올리기 때문이다. 실측 영분포는 이 셋을 전부 포함한다. BH 는 비교 방법끼리 같은 자로 잘라야 할 때(RQ1)를 위한 참고로만 둔다.

→ 이것이 계획서 7.2 의 **"무주입 거짓 경보율"** 이고, 주입 실험의 **영분포**로 씀.
→ 동시에 이 방법의 **탐지 하한**임. 무료 유저처럼 모델이 못 맞추는 집단에서는 6%p대 잔차가 남음.

### 고친 것 ① — 기준선의 오차를 t 에 넣음

`하락 = 기준선 − 평가 편차` 인데 **기준선도 추정값**임. 그 분산을 빼먹으면 "기준선을 정확히 안다"고
가정하는 셈이라 t 가 크게 부풀음.

| | 무시 | **넣음** |
|---|---|---|
| 최고 t | 16.49 | **7.47** |
| t > 4 인 후보 | 4,436개 | **468개** (−89%) |
| 1위 가설 | 15만 명 × 2.05%p (보정 잔차가 큰 **큰** 집단) | 1.6만 명 × 6.60%p |
| 습관 집단 t — 미측정 / 4-10일 / 11-20일 | 6.2 / 7.3 / 3.4 | **1.0 / 1.2 / 0.9** |

**"큰 집단의 작은 잔차" 가 ±기준선에 묻혀 더는 올라오지 못함.** 기울기를 잘 못 재는 집단이
자동으로 불리해지므로, `Modeling.ipynb` 6-5 에서 게이트·축소로 억지로 하려다 실패한 일이
표준오차 하나로 해결됨.

### 고친 것 ② — 사실상 아무것도 안 거르는 조건 차단

처음 돌렸을 때 상위 20개가 전부 `act_days ≤10 AND days_since_last >0` 에
`is_cancel ≠ (결측)` 같은 **98%를 그대로 남기는 조건**을 덧붙인 것이었음. t 를 소수점 둘째 자리만
올리며 목록을 채움. → 집단이 **5% 미만으로 줄면 버림**.

### 절제 실험 재료 — 기준선 보정을 빼면

| | 1위 가설 | 인원 | 평가 편차 | 하락 | t |
|---|---|---|---|---|---|
| **보정 X** (전체 기준선만) | `act_days 21-28일` AND `days_since_last 0일` AND `weekend_ratio ≤0.32` | 300,737 | **−0.25** | +1.71 | **33.9** |
| **보정 O** | `act_days ≤10` AND `days_since_last >0` AND `trend ≤3` | 155,289 | +5.92 | +2.05 | 16.5 |

보정을 빼면 **헤비 유저 집단이 1위**가 됨. 평가 편차가 −0.25 로 거의 0 인 **정상 집단**인데,
전체 기준선 +1.46%p 를 들이대니 "1.71%p 하락" 으로 둔갑함. 전체 기준선은 **가벼운 유저 때문에
올라간 값**이라, 전체의 절반을 차지하는 헤비 집단에 그대로 적용하면 통째로 가짜 하락이 됨.
→ ①안(전체 기준선만 쓰기)의 치명적 약점을 수치로 보인 사례. 계획서 7.3 절제 실험에 쓸 것.

### ⚠️ 구현 메모 — 계획과 달라진 점

계획서·설계 문서에는 **`pysubgroup` 활용**이라 적혀 있으나 **직접 구현했음.**

| 이유 | |
|---|---|
| 점수 함수 | 기본 품질 함수는 "집단 평균 vs 전체 평균" 형태임. 우리 점수는 **정상 기간에서 회귀를 적합해 기준선을 만들고 그 오차까지 분모에 넣는** 것이라 어차피 커스텀이 필요함 |
| 속도 | 후보마다 행을 걸러내면 1,721만 행 × 6,000 후보가 되어 몇 시간. 지금은 **축 하나에 조건 전부를 `bincount` 로 한 번에** 계산해 1분 |

**비표준으로 추가한 것 둘** (계획에 없던 것):
① 같은 축을 두 번 쓰지 않음 (표준 빔 서치는 `act_days ≤10 AND act_days >3` 을 허용함)
② 집단이 5% 미만으로 줄면 버림

**남은 과제 — 교차 검증.** `pysubgroup` 을 **표준 품질 함수**로 같은 조건·같은 설정에 돌려
상위 집단이 일치하는지 확인할 것. 검증되는 것은 **탐색 기계장치**(가지치기·확장·순위)이고,
점수 함수는 이미 `Modeling.ipynb` 6-3 의 ±보정과 대조해 일치를 확인했음
(미측정 0.840 vs 0.805 / 1-3일 0.685 vs 0.656 / 21-28일 0.123 vs 0.118).
10/8 시도는 API 인자 순서·반환 형식이 달라 중단함.

### 다음 — 구현 5번: 주입 벤치마크

- 여기서 나온 **최고 t = 7.47** 이 무주입 영분포의 기준점
- 정답 집단에 δ 를 주입해 **δ 를 얼마나 키워야 이 기준선을 넘는지** 곡선으로 측정
- 정답 집단의 **크기를 바꿔 가며** 탐지 하한도 함께 (1만 행 미만은 전체 기준선만 쓰므로 불리)

## 7. 집단 하나 들여다보기 (점검 도구)

빔 서치는 집단 수천 개를 **점수 하나로 줄여** 순위만 보여 줌. 이 절은 반대로 **집단 하나를 골라
그 점수가 어떻게 만들어졌는지** 중간값을 전부 펼쳐 보는 도구임.

### 쓰는 법

```python
조건찾기('age')                          # 쓸 수 있는 조건 이름 찾기
점검('age = 20s')                        # 중간값 전부
점검('is_cancel = 예', 'act_days ≤10')   # 조건 2~3개 (같은 축은 두 번 쓰지 말 것)
점검('act_days 1-3일', 날짜별=True)      # 기준일별 평균 편차까지
```

2·3·4절이 실행돼 있어야 함. **5절(빔 서치)은 안 돌려도 됨.**

### 보는 법

| 줄 | 뜻 |
|---|---|
| 행 / 유효 인원 | 유효 인원은 **유저별 평가 행 수의 역수 합**. 전체 유저 수가 아님 — 집단이 그 유저의 평가 행 일부만 담으면 그만큼만 셈 |
| 날짜 / gap | `gap = 평가 평균날짜 − 학습 평균날짜`. **집단마다 다름** (전체 14.46일, 월말에 행이 몰린 집단은 16일) |
| 학습 | 직선을 적합한 구간의 평균 편차와 기울기 |
| 기준선 ± | 그 직선을 평가 기간까지 연장한 값과 **그 값의 오차** |
| 평가 ± | 평가 기간 실제 편차와 그 표준오차 (유효 인원 기준) |
| 하락 / t | `하락 = 기준선 − 평가 편차`, `t = 하락 ÷ √(두 오차의 제곱합)` |
| 대조 | 같은 합을 4절 `score()` 에 그대로 넣어 **빔 서치와 같은 값이 나오는지** 확인 |
| 참고 | 보정 없이 **전체 기준선(+1.462%p)만** 썼다면 어땠을지 |

### 실행 예 — 집단 네 개 (단위 %p)

| 집단 | 유효 인원 | 기울기 | 기준선 | 평가 편차 | 하락 | t | **보정 없으면 t** |
|---|---|---|---|---|---|---|---|
| `age = 20s` | 264,755 | +0.085 | +1.242 ±0.269 | +1.288 | −0.045 | −0.16 | **+2.50** |
| `is_cancel = 예` | 9,562 | +0.112 | +1.821 ±0.644 | +1.341 | +0.480 | +0.63 | +0.30 |
| `act_days 1-3일` | 63,081 | +0.446 | +6.736 ±0.685 | +6.763 | −0.027 | −0.04 | **−28.09** |
| 1위 가설 (조건 3개) | 16,302 | +0.361 | +6.355 ±0.801 | −0.240 | **+6.595** | **+7.47** | +4.59 |

읽을 거리 셋.

- **`age = 20s`** — 보정이 없으면 t +2.50 으로 떠오르지만, 이 집단의 기준선이 +1.242 라 실제로는 조용함.
  **거짓 양성이 걸러진 예**임
- **`act_days 1-3일`** — 보정이 없으면 t **−28.09**. 전체 기준선 +1.46 을 들이대면 "크게 상승한 집단"으로
  보이는데, 제 기준선은 +6.74 라 사실은 제자리임. **자를 집단마다 따로 들어야 하는 이유**
- **`is_cancel = 예`** — 보정이 신호를 **키운** 경우(+0.30 → +0.63). 보정은 늘 깎는 방향이 아님

In [7]:
_t0 = time.time()

# ── 쓰는 법 ─────────────────────────────────────────────────────────────
#   조건찾기('age')                        쓸 수 있는 조건 이름을 찾음
#   점검('age = 20s')                      중간값을 전부 찍음
#   점검('is_cancel = 예', 'act_days ≤10') 조건 2~3개도 됨 (같은 축은 두 번 쓰지 말 것)
#   점검('act_days 1-3일', 날짜별=True)    기준일별 평균 편차까지 봄
# 2·3·4절이 실행돼 있어야 함. 5절(빔 서치)은 안 돌려도 됨


def 조건찾기(키워드=''):
    """조건 이름에 키워드가 들어간 것만 보여 줌 — 이름을 정확히 알아야 점검() 에 넣을 수 있음"""
    hit = [c['이름'] for c in CONDS if 키워드.lower() in c['이름'].lower()]
    print('조건 {:,}개 중 {:,}개 일치'.format(len(CONDS), len(hit)))
    for i in range(0, len(hit), 3):
        print('   ' + '  '.join('{:<32s}'.format(x) for x in hit[i:i + 3]).rstrip())


def 점검(*이름들, 날짜별=False):
    """집단 하나의 점수가 만들어지는 중간값을 전부 찍음 — 4절 score() 와 같은 규칙으로 계산함"""
    # ── 1) 조건을 차례로 걸어 행을 고름 (빔 서치가 하는 것과 같음) ───────
    idx = np.arange(N)
    for nm in 이름들:
        c = next((x for x in CONDS if x['이름'] == nm), None)
        if c is None:
            print('조건을 못 찾음: {!r}   →  조건찾기({!r}) 로 이름을 확인할 것'.format(
                nm, nm.split(' ')[0]))
            return
        idx = idx[c['sel'][CODE[c['축']][idx]]]

    k = np.searchsorted(idx, N_TR)
    it, ie = idx[:k], idx[k:]
    n, ne = len(it), len(ie)
    print('집단: {}   (조건 {}개)'.format(' AND '.join(이름들) if 이름들 else '전체', len(이름들)))
    if ne < 1 or n < 2:
        print('  행이 너무 적어 잴 수 없음 — 학습 {:,} / 평가 {:,}'.format(n, ne))
        return

    # ── 2) 합 12개 — 빔 서치가 후보마다 세는 것과 같은 것들 ──────────────
    d_t, v_t = DAY[it], DEV[it]
    d_e, v_e = DAY[ie], DEV[ie]
    Sd, Sv, Sdd, Sdv = d_t.sum(), v_t.sum(), (d_t * d_t).sum(), (d_t * v_t).sum()
    Su = WU[ie].sum()                                        # 유효 인원 (유저 중복 보정)
    e_tr, d_tr = Sv / n, Sd / n
    e_ev, d_ev = v_e.mean(), d_e.mean()
    Gn = np.bincount(DAYI[it], minlength=NDAY).astype('float64')
    Gv = np.bincount(DAYI[it], weights=v_t, minlength=NDAY)

    # ── 3) 기준선 — 4절 score() 와 같은 조건으로 집단별/전체를 가름 ──────
    Sxx, ok = Sdd - Sd * Sd / n, Gn > 0
    gap, 날짜수 = d_ev - d_tr, int(ok.sum())
    집단별 = (ne >= MIN_FIT) and (Sxx > 0) and (날짜수 >= 3)
    if 집단별:
        b = (Sdv - Sd * Sv / n) / Sxx
        a = e_tr - b * d_tr
        base = a + b * d_ev
        r = Gv[ok] / Gn[ok] - (a + b * DAYS[ok])             # 날짜별 평균이 직선에서 벗어난 정도
        s2 = (Gn[ok] * r * r).sum() / max(날짜수 - 2, 1)
        var_base = s2 * (1.0 / n + gap * gap / Sxx)
        단서 = '← 이 집단의 직선으로 구함'
    else:
        b, base, var_base = 0.0, BASE_ALL, 0.0
        단서 = '← 전체 기준선 (평가 {:,}행 < {:,})'.format(ne, MIN_FIT) if ne < MIN_FIT \
               else '← 전체 기준선 (학습 날짜 {}개 < 3)'.format(날짜수)
    var_ev = max(v_e.var(), 1e-12)
    drop = base - e_ev
    t = drop / np.sqrt(var_ev / Su + var_base)

    # ── 4) 출력 ─────────────────────────────────────────────────────────
    print('  행        학습 {:>10,} / 평가 {:>10,}      유효 인원 {:,.0f}명{}'.format(
        n, ne, Su, '  ⚠️ 최소 {}명 미만이라 빔 서치는 버림'.format(MIN_USERS)
        if Su < MIN_USERS else ''))
    print('  날짜      학습 평균 {:.2f} / 평가 평균 {:.2f}      gap {:.2f}일   (학습 날짜 {}개)'.format(
        d_tr, d_ev, gap, 날짜수))
    print('  학습      평균 편차 {:+.3f}%p      기울기 {:+.4f}%p/일'.format(e_tr * 100, b * 100))
    print('  기준선    {:+.3f}%p  ± {:.3f}      {}'.format(
        base * 100, np.sqrt(var_base) * 100, 단서))
    print('  평가      실제 편차 {:+.3f}%p  ± {:.3f}'.format(
        e_ev * 100, np.sqrt(var_ev / Su) * 100))
    print('  ' + '─' * 62)
    print('  하락 {:+.3f}%p       t = {:+.2f}       잃은 유저 {:+,.0f}명'.format(
        drop * 100, t, drop * Su))

    # 같은 합을 4절 score() 에 그대로 넣어 결과가 같은지 확인함 (점검 셀이 빔 서치와 어긋나면 안 됨)
    r2 = score(np.array([n, Sd, Sv, Sdd, Sdv]),
               np.array([ne, Su, v_e.sum(), (v_e * v_e).sum(), d_e.sum()]), Gn, Gv)
    맞음 = r2 is not None and abs(r2[2] - t) < 1e-6
    print('  대조      빔 서치 score() → {}'.format(
        '하락 {:+.3f} · t {:+.2f} · ±기준선 {:.3f} · 기울기 {:+.4f}   {}'.format(
            r2[1], r2[2], r2[3], r2[4], '일치 ✓' if 맞음 else '⚠️ 어긋남') if r2 is not None
        else 'None (최소 인원 미달이라 후보에서 제외되는 집단임)'))
    print('  참고      보정 없이 전체 기준선({:+.3f})만 쓰면  하락 {:+.3f}%p · t {:+.2f}'.format(
        BASE_ALL * 100, (BASE_ALL - e_ev) * 100,
        (BASE_ALL - e_ev) / np.sqrt(var_ev / Su)))

    # ── 5) 날짜별 평균 편차 (선택) ───────────────────────────────────────
    if 날짜별:
        ce = np.bincount(d_e.astype('int32'), minlength=NDAY + 14).astype('float64')
        ve = np.bincount(d_e.astype('int32'), weights=v_e, minlength=NDAY + 14)
        print('  기준일별 평균 편차 (%p)')
        for 라벨, G_, V_, 구간 in [('    학습', Gn, Gv, range(NDAY)),
                                   ('    평가', ce, ve, range(NDAY, len(ce)))]:
            print(라벨 + ' ' + ' '.join(
                '{:>6.2f}'.format(V_[i] / G_[i] * 100) if G_[i] > 0 else '     ·' for i in 구간))


print('7절 준비 완료 — 조건찾기() · 점검() 를 쓸 수 있음   ({:.0f}초)'.format(time.time() - _t0))


# ── 보기 — 이 아래 줄을 고쳐 가며 쓰면 됨 ──────────────────────────────
#   조건찾기('payment')  /  점검('age = 20s')  /  점검('is_cancel = 예', 'act_days ≤10')
점검('act_days 1-3일', 날짜별=True)

7절 준비 완료 — 조건찾기() · 점검() 를 쓸 수 있음   (0초)
집단: act_days 1-3일   (조건 1개)
  행        학습    231,898 / 평가    240,360      유효 인원 63,081명
  날짜      학습 평균 6.98 / 평가 평균 21.52      gap 14.53일   (학습 날짜 15개)
  학습      평균 편차 +0.257%p      기울기 +0.4458%p/일
  기준선    +6.736%p  ± 0.685      ← 이 집단의 직선으로 구함
  평가      실제 편차 +6.763%p  ± 0.189
  ──────────────────────────────────────────────────────────────
  하락 -0.027%p       t = -0.04       잃은 유저 -17명
  대조      빔 서치 score() → 하락 -0.027 · t -0.04 · ±기준선 0.685 · 기울기 +0.4458   일치 ✓
  참고      보정 없이 전체 기준선(+1.462)만 쓰면  하락 -5.300%p · t -28.09
  기준일별 평균 편차 (%p)
    학습  -3.18  -1.96  -0.93  -1.60  -1.91  -1.76  -0.77   0.31   2.07   1.98   1.79   1.57   1.77   2.44   4.12
    평가   4.53   5.72   5.45   5.09   5.26   6.34   7.69   8.26   6.68   6.36   7.03   8.95   8.45   9.01


In [8]:
점검('act_days 1-3일', 날짜별=True)

집단: act_days 1-3일   (조건 1개)
  행        학습    231,898 / 평가    240,360      유효 인원 63,081명
  날짜      학습 평균 6.98 / 평가 평균 21.52      gap 14.53일   (학습 날짜 15개)
  학습      평균 편차 +0.257%p      기울기 +0.4458%p/일
  기준선    +6.736%p  ± 0.685      ← 이 집단의 직선으로 구함
  평가      실제 편차 +6.763%p  ± 0.189
  ──────────────────────────────────────────────────────────────
  하락 -0.027%p       t = -0.04       잃은 유저 -17명
  대조      빔 서치 score() → 하락 -0.027 · t -0.04 · ±기준선 0.685 · 기울기 +0.4458   일치 ✓
  참고      보정 없이 전체 기준선(+1.462)만 쓰면  하락 -5.300%p · t -28.09
  기준일별 평균 편차 (%p)
    학습  -3.18  -1.96  -0.93  -1.60  -1.91  -1.76  -0.77   0.31   2.07   1.98   1.79   1.57   1.77   2.44   4.12
    평가   4.53   5.72   5.45   5.09   5.26   6.34   7.69   8.26   6.68   6.36   7.03   8.95   8.45   9.01


## 8. 주입 벤치마크 (구현 5번)

### 왜 하는가

실제 데이터에는 하락이 없다 (평가 기간이 학습보다 오히려 +0.40%p). 그래서 "이 방법이 하락을 찾아내는가" 를 그냥은 시험할 수 없다.
→ **정답을 아는 하락을 직접 심는다.** 시험 문제를 우리가 내고, 답을 알고 있고, 채점한다.

### 다섯 단계

| | 하는 일 |
|---|---|
| ① | 정답 집단을 하나 고른다 (무주입에서 조용한 집단) |
| ② | 그 집단의 **평가 기간** 행에서 "돌아왔다" 를 "안 돌아왔다" 로 δ%p 만큼 바꾼다 |
| ③ | 빔 서치를 **아무 말 없이** 다시 돌린다 |
| ④ | 빔 서치가 내놓은 상위 10개가 ①과 같은 집단인지 자카드로 채점한다 |
| ⑤ | δ 를 키우며 반복 → **얼마나 떨어져야 찾아내나** 곡선 |

### 합격선은 무주입 최고 t = 7.47

6절에서 본 대로, 하락이 하나도 없는 데이터에서도 후보 6,000개를 뒤지면 최고 t 가 7.47 까지 나온다 (순수 잡음).
주입한 하락은 **이 선을 넘어야** 발견된 것으로 친다. 이 선이 곧 이 방법의 **탐지 하한**이다.

### 설계상 지켜지는 것

- 주입은 **평가 기간 행에만** 한다 → 학습(정상) 기간이 그대로라 기대 모델도 기준선 기울기도 다시 구하지 않는다 (계획서 7.2 "모델은 한 번만 학습")
- 재방문 `d1` 을 1 → 0 으로 뒤집으면 편차 `dev = 실제 − p` 가 정확히 **1.0 낮아진다** → `DEV` 배열만 고치면 된다
- 정답 집단은 **조건 1개로 표현되는 것**만 쓴다 → 빔 서치가 1단계에서 바로 찾을 수 있어야 공정하다

⚠️ **먼저 2·3·4·5절을 실행해 둘 것.** 5절은 무주입 기준선(최고 t 7.47)을 화면에 띄우는 데 쓴다.

### 8-1. 주입 도구와 재현 점검

탐색 자체(`빔서치()` · `추려내기()`)는 **5절에서 이미 정의**했다. 여기서는 주입 쪽만 준비한다.

| 이름 | 하는 일 |
|---|---|
| `DEV0` | 주입 전 **원본 편차 배열**. 모든 주입이 여기서 출발한다 |
| `주입(집단, δ, seed)` | 집단의 평가 행에서 복귀한 사람 일부를 0 으로 뒤집은 **새 편차 배열**을 돌려준다 |
| `원복()` | 전역 `DEV` 를 원본으로 되돌린다 (주입 실험 뒤 뒷정리) |

**뒤집는 개수 = δ/100 × 그 집단의 평가 행 수** 라서, 집단 평균 복귀율이 정확히 δ%p 내려간다.

> 예) 28,791행짜리 집단에 δ = 5%p → **1,440행**을 1→0 → 복귀율 0.795 → 0.745

### 마지막 점검

주입하지 않은 원본으로 `빔서치()` 를 한 번 돌려 5절과 같은 답(**최고 t 7.47 · 1위 16,302명**)이
나오는지 확인하고, 다르면 `assert` 로 멈춘다. 8절의 모든 결과가 이 보증 위에 있다.

In [9]:
# ── 8-1. 주입 도구 · 무주입 재현 점검 ───────────────────────────────────
_t0 = time.time()
assert '빔서치' in globals(), '5절을 먼저 실행할 것 — 빔서치()·추려내기() 를 가져다 씀'

# 원본 편차는 4절 배열에서 다시 만듦 (전역 DEV 는 앞선 주입으로 바뀌어 있을 수 있음)
DEV0 = dev[_keep].astype('float64').copy()
D1   = pred['d1'].to_numpy().astype('float64')[_keep]   # 재방문 여부
EV   = np.zeros(N, dtype=bool); EV[N_TR:] = True        # 평가 기간 행
# dev = 실제 − p 이므로 d1 을 1에서 0 으로 뒤집으면 dev 가 정확히 1.0 낮아짐
assert np.allclose(DEV0, D1 - pred['p'].to_numpy().astype('float64')[_keep]), 'DEV 와 d1-p 가 어긋남'

print('평가 기간 복귀율 {:.4f} / 복귀 행 {:,}개'.format(D1[N_TR:].mean(), int(D1[N_TR:].sum())))


def 주입(sel, delta, seed):
    """집단(sel, N칸 불리언)의 **평가 기간 행**에서 복귀를 delta(%p) 만큼 0 으로 뒤집음.
       뒤집을 행 수 = delta/100 x (집단의 평가 행 수) 이므로 집단 평균 d1 이 정확히 그만큼 내려감.
       복귀한 사람이 모자라면 None 을 돌려줌 (그 집단에서 그 크기의 하락은 물리적으로 불가능함)"""
    ne   = int((sel & EV).sum())
    pool = np.flatnonzero(sel & EV & (D1 > 0.5))
    k    = int(round(delta / 100.0 * ne))
    if k > len(pool):
        return None
    d = DEV0.copy()
    d[np.random.default_rng(seed).choice(pool, k, replace=False)] -= 1.0
    return d


def 원복():
    """전역 DEV 를 주입 전 원본으로 되돌림 (주입 실험 뒤 뒷정리)"""
    global DEV, _WT, _WE, BASE_ALL
    DEV, BASE_ALL = DEV0, DEV0[N_TR:].mean()
    _WT = [None, DAY, DEV, DAY * DAY, DAY * DEV]
    _WE = [None, WU,  DEV, DEV * DEV, DAY]


# ── 점검: 무주입으로 돌려 5절 결과(최고 t 7.47 · 1위 16,302명)를 재현하는지 ──
_a0    = 빔서치(DEV0)
_top0  = 추려내기(_a0, 1)[0]
T_NULL = _a0[0][0][2]                               # 영분포 기준점 — 8-3·8-4 에서 씀
print('\n[점검] 무주입 재현 — 최고 t {:.2f} / 1위 {:,.0f}명 / 하락 {:+.2f}%p'.format(
    T_NULL, _top0[0][0], _top0[0][1]))
print('       1위 가설: {}'.format(' AND '.join(c['이름'] for c in _top0[1])))
assert abs(T_NULL - 7.47) < 0.05, '5절과 다름 — 데이터나 점수 함수가 바뀌었는지 확인할 것'
assert abs(_top0[0][0] - 16302) < 5, '1위 집단 인원이 5절과 다름'
print('       5절과 일치 (OK)   ({:.0f}초)'.format(time.time() - _t0))

평가 기간 복귀율 0.7852 / 복귀 행 6,601,765개

[점검] 무주입 재현 — 최고 t 7.47 / 1위 16,302명 / 하락 +6.60%p
       1위 가설: payment_plan_days 결측 AND channel ≠ 4 AND act_days ≤20
       5절과 일치 (OK)   (36초)


### 8-2. 정답 집단은 어떻게 고르나

조건 1개짜리 집단 286개를 전부 채점해 놓고 세 가지를 본다.

| 기준 | 이유 |
|---|---|
| **조건 1개로 표현될 것** | 빔 서치가 1단계에서 바로 찾을 수 있어야 공정하다 |
| **무주입에서 조용할 것** (t 의 절댓값 < 1) | 이미 거짓 경보로 떠 있는 집단에 주입하면 공이 누구 것인지 알 수 없다 |
| **크기 3단계** | 아래 참고 |

### 왜 크기를 3단계로 나누나 — 1만 행이 경계다

설계 3절에서 **평가 행 1만 이상인 집단만 집단별 기준선 보정**을 받기로 했다 (`MIN_FIT`). 그 아래는 전체 기준선을 쓴다.

```
평가 1만 행 미만  →  집단별 보정 못 받음  →  탐지에 불리할 것   ← 이걸 재려는 것
평가 1만 행 이상  →  집단별 보정 받음
```

설계 3절에 한계로 적어 둔 **"1만 행 미만 집단의 진짜 하락은 놓칠 수 있다(거짓 음성)"** 를
수치로 보이는 것이 이 실험의 목적 중 하나다.

⚠️ 후보 표를 `payment_method_id` 가 뒤덮는다 — 조건 286개의 과반이 결제수단·요금제기간 **코드 값**이기 때문
(10/6 조건 수 점검에서 이미 나온 문제). 그래서 같은 크기대 안에서는 가능한 한 해석되는 축을 골랐다.

In [10]:
# ── 8-2. 정답 집단 고르기 ───────────────────────────────────────────────
_t0 = time.time()

# 고르는 기준 세 가지
#   (1) **조건 1개로 표현되는 집단** — 빔 서치가 1단계에서 바로 찾을 수 있어야 공정함
#   (2) **무주입에서 조용할 것**(|t| 작음) — 이미 거짓 경보로 떠 있는 집단에 주입하면 공을 가릴 수 없음
#   (3) **크기 3단계** — 보정 임계값 MIN_FIT(평가 1만 행)을 걸치게 잡음.
#       그 아래 집단은 집단별 기준선을 못 쓰고 전체 기준선을 쓰므로 탐지에 불리함 (설계 3절의 한계)
TARGETS = [3_000, 30_000, 300_000]      # 노리는 평가 행 수
QUIET   = 1.0                           # 무주입 |t| 가 이보다 작아야 '조용한' 집단

_S0, _rec = cell_sums(_root), []
for _c in CONDS:
    _a, _ax = _c['sel'], _c['축']
    _T, _E = _S0[_ax][0][:, _a].sum(1), _S0[_ax][1][:, _a].sum(1)
    _r = score(_T, _E, _S0[_ax][2][_a].sum(0), _S0[_ax][3][_a].sum(0))
    if _r is None:
        continue
    _sel = _c['sel'][CODE[_ax]]
    _rec.append({'이름': _c['이름'], '평가행': int(_E[0]), '유효인원': _r[0],
                 '복귀율': D1[EV & _sel].mean(), '무주입t': _r[2]})
_cand = pd.DataFrame(_rec)

print('조건 1개짜리 채점 가능 집단 {}개   (평가 행 {:,} ~ {:,})'.format(
    len(_cand), _cand['평가행'].min(), _cand['평가행'].max()))
print('\n크기대별 후보 — 무주입 |t| < {} 인 조용한 집단만'.format(QUIET))
_fmt = {'평가행': '{:,.0f}'.format, '유효인원': '{:,.0f}'.format,
        '복귀율': '{:.3f}'.format, '무주입t': '{:+.2f}'.format}
for _tg in TARGETS:
    _q = _cand[_cand['무주입t'].abs() < QUIET].copy()
    _q['거리'] = np.abs(np.log(_q['평가행'] / _tg))
    print('\n  -- 평가 행 {:,} 근처   {} --'.format(
        _tg, '보정 안 함 (< {:,})'.format(MIN_FIT) if _tg < MIN_FIT else '집단별 보정'))
    print(_q.nsmallest(5, '거리')[['이름', '평가행', '유효인원', '복귀율', '무주입t']]
            .to_string(index=False, formatters=_fmt))
print('\n{:.0f}초 — 크기대별로 하나씩 골라 8-3 의 GOLD 에 적을 것'.format(time.time() - _t0))

조건 1개짜리 채점 가능 집단 286개   (평가 행 1,284 ~ 8,407,744)

크기대별 후보 — 무주입 |t| < 1.0 인 조용한 집단만

  -- 평가 행 3,000 근처   보정 안 함 (< 10,000) --
                    이름   평가행 유효인원   복귀율  무주입t
payment_method_id = 26 2,973  325 0.795 +0.71
payment_method_id = 14 3,749  400 0.803 -0.54
payment_method_id = 18 4,381  511 0.776 -0.11
 payment_method_id = 8 1,673  177 0.819 +0.36
 payment_method_id = 6 1,551  176 0.770 +0.50

  -- 평가 행 30,000 근처   집단별 보정 --
                    이름    평가행  유효인원   복귀율  무주입t
       channel = other 28,791 3,358 0.795 +0.35
payment_method_id = 12 24,333 2,524 0.819 -0.12
payment_method_id = 13 50,688 4,854 0.848 +0.27
payment_method_id = 23 17,001 1,865 0.791 +0.16
payment_method_id = 15 56,567 5,274 0.860 +0.36

  -- 평가 행 300,000 근처   집단별 보정 --
                    이름     평가행   유효인원   복귀율  무주입t
payment_method_id = 37 322,265 34,856 0.805 +0.27
         act_days 1-3일 240,360 63,081 0.507 -0.04
             age = 10s 443,020 45,632 0.849 +0.82
payment_method_id = 29 176,995 20,731 0.79

### 8-3. 결과표 읽는 법

| 칸 | 뜻 |
|---|---|
| `최고t` | **그 판 1등의 점수.** 7.47(무주입 최고)보다 커야 "잡음과 구별되는 신호" |
| `정답t` | 우리가 손댄 집단 **자신의** 점수. 1등이 아니어도 몇 점인지 보려고 |
| `순위` | 상위 10개 중 정답과 행이 **50% 이상 겹치는** 첫 가설의 순위. `-` 는 10개 안에 없다는 뜻 |
| `최고자카드` | 상위 10개 중 정답과 가장 많이 겹친 정도 (1.00 = 글자 그대로 일치) |

### 한 줄 예시

```
소 δ=12%p   최고t 7.47   정답t 5.89   순위 2   자카드 1.00
```

- 12%p 를 떨어뜨렸다
- 그런데 그 판 1등 점수가 **7.47 — 주입 전과 똑같다** (1등은 주입과 무관한 기존 거짓 경보)
- 우리가 손댄 집단은 5.89점으로 **2등**
- 찾아낸 집단은 정답과 **100% 일치** (자카드 1.00)

→ **정확히 찾아내고도 점수가 모자라 내놓지 못한 것.** 이것이 "거짓 음성" 의 실제 모습이다.

### 지표 주의 — Hit@1 은 큰 집단에 불리하다

큰 집단은 조건을 더 붙여 **부분집합**을 만들어도 여전히 크기 때문에, 그 부분집합이 더 높은 t 로 1등을 가져간다
(대 집단의 자카드가 0.57~0.73 에서 멈추는 이유). 반대로 작은 집단은 조건을 더 붙이면 100명 미만이 되어
원형 그대로 나온다 (자카드 1.00).

→ **Hit@3 과 최고자카드를 주 지표로 보고, Hit@1 은 참고로 본다.** 계획서 7.2 가 이미 Hit@1·3·5 와
최고 Jaccard 를 모두 보고하기로 하고 있으므로 설계 변경이 아니라 **해석 주의사항**이다.

In [11]:
# ── 8-3. 탐지 하한 곡선 — 파일럿 ────────────────────────────────────────
_t0 = time.time()

# 정답 집단 3개 (8-2 에서 고름). 크기만 다르고 셋 다 무주입에서 조용함(|t| < 0.4)
#   소 — 평가 4,381행: MIN_FIT(1만) 아래라 **집단별 보정을 못 받고 전체 기준선**을 씀 → 불리
#   중 — 평가 28,791행: 보정 받음. 셋 중 유일하게 해석이 쉬운 축(가입 경로)
#   대 — 평가 522,068행: 보정 받음
# 뺀 것: age = 10s (무주입 t +0.82 로 이미 올라와 있음),
#        act_days 1-3일 (기울기 +0.45%p/일 로 보정이 세게 걸리는 특수 케이스 → 8-5 에서 따로)
GOLD = [('소', 'payment_method_id = 18'),
        ('중', 'channel = other'),
        ('대', 'city = city 15')]
DELTAS = [1, 2, 3, 5, 8, 12, 18, 25]        # 주입할 하락 폭 (%p)

PILOT_PATH = 'data/bench_pilot.csv'
RUN_PILOT  = True        # 한 번 돌린 뒤 False 로 두면 저장된 결과를 불러씀 (Run All 사고 방지)

_byname = {c['이름']: c for c in CONDS}
for _, _nm in GOLD:
    assert _nm in _byname, '조건 이름을 못 찾음: {}'.format(_nm)


def 집단마스크(name):
    """조건 이름 → N칸 불리언"""
    c = _byname[name]
    return c['sel'][CODE[c['축']]]


def 집단점수(idx, dev_use):
    """집단 하나를 직접 채점함 (4절 score 와 같은 합 12개).
       ⚠️ 호출 전에 빔서치(dev_use) 가 돌아 전역 BASE_ALL 이 맞춰져 있어야 함"""
    k = np.searchsorted(idx, N_TR)
    it, ie = idx[:k], idx[k:]
    T = np.array([len(it), DAY[it].sum(), dev_use[it].sum(),
                  (DAY[it] ** 2).sum(), (DAY[it] * dev_use[it]).sum()])
    E = np.array([len(ie), WU[ie].sum(), dev_use[ie].sum(),
                  (dev_use[ie] ** 2).sum(), DAY[ie].sum()])
    Gn = np.bincount(DAYI[it], minlength=NDAY).astype('float64')
    Gv = np.bincount(DAYI[it], weights=dev_use[it], minlength=NDAY)
    return score(T, E, Gn, Gv)


def 자카드(a, b):
    return len(np.intersect1d(a, b, True)) / len(np.union1d(a, b))


if RUN_PILOT:
    _rows, _n = [], len(GOLD) * len(DELTAS)
    for _크기, _nm in GOLD:
        _sel  = 집단마스크(_nm)
        _gidx = np.flatnonzero(_sel)
        for _d in DELTAS:
            _dev = 주입(_sel, _d, seed=20261009)
            if _dev is None:                       # 복귀한 사람이 모자라 주입 불가
                continue
            _allc = 빔서치(_dev)                    # 전역 DEV·BASE_ALL 이 여기서 바뀜
            _top  = 추려내기(_allc, TOPK)
            _gr   = 집단점수(_gidx, _dev)           # 정답 집단 자신의 점수

            # 상위 10개 중 정답과 행이 50% 이상 겹치는 첫 가설을 찾음 (설계 7.2)
            _rank, _jac = 0, 0.0
            for _i, (_r, _desc) in enumerate(_top, 1):
                _j = 자카드(rows(_desc), _gidx)
                if _j > _jac:
                    _jac = _j
                if _j >= MAX_JACC and _rank == 0:
                    _rank = _i
            _rows.append({'크기': _크기, '집단': _nm, 'δ': _d,
                          '최고t': _allc[0][0][2], '정답t': _gr[2], '정답하락': _gr[1],
                          '순위': _rank, '최고자카드': _jac,
                          '영분포넘음': _allc[0][0][2] > T_NULL,
                          'Hit@1': int(_rank == 1), 'Hit@3': int(0 < _rank <= 3),
                          'Hit@5': int(0 < _rank <= 5)})
            print('  {} δ={:>2}%p  최고t {:>6.2f}  정답t {:>6.2f}  순위 {}  자카드 {:.2f}   ({:.0f}초/{} 중 {})'
                  .format(_크기, _d, _rows[-1]['최고t'], _rows[-1]['정답t'],
                          _rows[-1]['순위'] or '-', _jac, time.time() - _t0, _n, len(_rows)))
    원복()
    pilot = pd.DataFrame(_rows)
    pilot.to_csv(PILOT_PATH, index=False, encoding='utf-8-sig')
    print('\n저장: {}   ({:.0f}초)'.format(PILOT_PATH, time.time() - _t0))
else:
    pilot = pd.read_csv(PILOT_PATH)
    print('불러옴: {} ({}행)'.format(PILOT_PATH, len(pilot)))

# ── 결과 ────────────────────────────────────────────────────────────────
print('\n영분포 기준점 T_NULL = {:.2f} (무주입 최고 t)'.format(T_NULL))
print(pilot[['크기', 'δ', '최고t', '정답t', '정답하락', '순위', '최고자카드', '영분포넘음']]
      .to_string(index=False, float_format='{:.2f}'.format))

print('\n크기별 탐지 임계 δ — 정답이 1위로 올라오고(Hit@1) 영분포를 넘은 가장 작은 δ')
for _크기, _nm in GOLD:
    _s = pilot[(pilot['크기'] == _크기) & (pilot['Hit@1'] == 1) & pilot['영분포넘음']]
    print('  {} ({:<26s}) — {}'.format(
        _크기, _nm, 'δ ≥ {}%p'.format(int(_s['δ'].min())) if len(_s) else '{}%p 까지 못 찾음'.format(max(DELTAS))))

  소 δ= 1%p  최고t   7.47  정답t   0.48  순위 -  자카드 0.00   (36초/24 중 1)
  소 δ= 2%p  최고t   7.47  정답t   1.04  순위 -  자카드 0.00   (73초/24 중 2)
  소 δ= 3%p  최고t   7.47  정답t   1.56  순위 -  자카드 0.00   (109초/24 중 3)
  소 δ= 5%p  최고t   7.47  정답t   2.59  순위 -  자카드 0.00   (145초/24 중 4)
  소 δ= 8%p  최고t   7.47  정답t   4.06  순위 -  자카드 0.00   (181초/24 중 5)
  소 δ=12%p  최고t   7.47  정답t   5.89  순위 2  자카드 1.00   (215초/24 중 6)
  소 δ=18%p  최고t   8.48  정답t   8.48  순위 1  자카드 1.00   (247초/24 중 7)
  소 δ=25%p  최고t  11.41  정답t  11.41  순위 1  자카드 1.00   (279초/24 중 8)
  중 δ= 1%p  최고t   7.47  정답t   1.34  순위 -  자카드 0.00   (316초/24 중 9)
  중 δ= 2%p  최고t   7.47  정답t   2.31  순위 -  자카드 0.00   (352초/24 중 10)
  중 δ= 3%p  최고t   7.47  정답t   3.27  순위 -  자카드 0.00   (388초/24 중 11)
  중 δ= 5%p  최고t   7.47  정답t   5.14  순위 2  자카드 0.60   (422초/24 중 12)
  중 δ= 8%p  최고t   8.39  정답t   7.86  순위 1  자카드 0.70   (451초/24 중 13)
  중 δ=12%p  최고t  11.62  정답t  11.37  순위 1  자카드 0.70   (480초/24 중 14)
  중 δ=18%p  최고t  16.70  정답t  16.47  순위 1  자카드 0.99   (510초/

### 8-4. 한 판 느리게 따라가기 (점검 도구)

주입 한 판에서 **숫자가 어떻게 만들어지는지** 중간값을 전부 찍는다. 7절 `점검()` 과 같은 성격의 도구다.

```
느린걸음('channel = other', 5)          # 중 집단에 5%p 주입
느린걸음('city = city 15', 2)           # 대 집단에 2%p
느린걸음('payment_method_id = 18', 12, 탐색=False)   # 빔 서치는 건너뜀 (즉시)
```

**보려는 것은 하나다** — 주입은 `평가 편차` 딱 하나만 바꾸고, 학습 평균·기울기·gap·기준선은 **전부 그대로**다.
학습 기간을 안 건드렸으니 당연한 일이지만, 눈으로 확인해 두면 "모델을 한 번만 학습한다" 는 설계가 왜 성립하는지 분명해진다.

- 2·3·4·5·8-1 절이 실행돼 있어야 한다. `탐색=True` 면 빔 서치를 한 번 돌려 약 40초 걸린다
- 마지막에 같은 값을 `집단점수()` 로 다시 구해 **손으로 푼 식과 일치하는지** 대조한다

In [12]:
# ── 8-4. 한 판 느리게 따라가기 ──────────────────────────────────────────
def _합(idx, dv):
    """학습 5개 + 평가 5개 + 날짜별 2개 — 4절 cell_sums 와 같은 것을 집단 하나에 대해서만"""
    k = np.searchsorted(idx, N_TR)
    it, ie = idx[:k], idx[k:]
    return (it, ie,
            np.bincount(DAYI[it], minlength=NDAY).astype('float64'),
            np.bincount(DAYI[it], weights=dv[it], minlength=NDAY))


def _풀이(idx, dv, base_all):
    """점수가 만들어지는 중간값을 전부 돌려줌 (4절 score 와 같은 식)"""
    it, ie, Gn, Gv = _합(idx, dv)
    n, ne = len(it), len(ie)
    Sd, Sv = DAY[it].sum(), dv[it].sum()
    Sxx = (DAY[it] ** 2).sum() - Sd * Sd / n
    dbar, dev_ev = Sd / n, DAY[ie].mean()
    o = {'학습행': n, '평가행': ne, '유효인원': WU[ie].sum(),
         '학습평균편차': Sv / n * 100, '평가평균편차': dv[ie].mean() * 100,
         '학습평균날짜': dbar, '평가평균날짜': dev_ev}
    ok = Gn > 0
    if ne >= MIN_FIT and Sxx > 0 and ok.sum() >= 3:
        b = ((DAY[it] * dv[it]).sum() - Sd * Sv / n) / Sxx
        a = Sv / n - b * dbar
        e = Gv[ok] / Gn[ok] - (a + b * DAYS[ok])
        s2 = (Gn[ok] * e * e).sum() / max(ok.sum() - 2, 1)
        o.update({'보정': '집단별', '기울기': b * 100, 'gap': dev_ev - dbar,
                  '기준선': (a + b * dev_ev) * 100, '날짜수': int(ok.sum()),
                  '기준선오차': np.sqrt(s2 * (1 / n + (dev_ev - dbar) ** 2 / Sxx)) * 100})
    else:
        o.update({'보정': '전체(평가 {:,}행 < {:,})'.format(ne, MIN_FIT), '기울기': 0.0,
                  'gap': dev_ev - dbar, '기준선': base_all * 100,
                  '날짜수': int(ok.sum()), '기준선오차': 0.0})
    o['하락'] = o['기준선'] - o['평가평균편차']
    o['평가오차'] = np.sqrt(max(np.mean(dv[ie] ** 2) - dv[ie].mean() ** 2, 1e-12) / o['유효인원']) * 100
    o['t'] = o['하락'] / np.sqrt(o['평가오차'] ** 2 + o['기준선오차'] ** 2)
    return o


def 느린걸음(이름, delta, 탐색=True, seed=20261009):
    t0 = time.time()
    sel, idx = 집단마스크(이름), np.flatnonzero(집단마스크(이름))
    ne   = int((sel & EV).sum())
    pool = np.flatnonzero(sel & EV & (D1 > 0.5))
    k    = int(round(delta / 100.0 * ne))

    print('=' * 78)
    print('[1] 집단 — {}'.format(이름))
    print('    평가 행 {:,}  /  그중 복귀한 행 {:,}  (복귀율 {:.4f})  /  유효 인원 {:,.0f}명'
          .format(ne, len(pool), len(pool) / ne, WU[idx[np.searchsorted(idx, N_TR):]].sum()))
    print('    보정 방식: {}'.format('집단별 기준선' if ne >= MIN_FIT else
                                 '전체 기준선 (평가 {:,}행 < {:,})'.format(ne, MIN_FIT)))

    print('\n[2] 뒤집기 — δ = {}%p'.format(delta))
    print('    뒤집을 행 수 = {}% x {:,} = {:,}행   (복귀한 {:,}행 중 {:.1%})'
          .format(delta, ne, k, len(pool), k / len(pool)))
    dv = 주입(sel, delta, seed)
    assert dv is not None, '복귀한 사람이 모자라 이만큼은 못 뒤집음'
    print('    집단 복귀율  {:.4f}  →  {:.4f}   (차이 {:+.2f}%p — 정확히 δ 만큼)'
          .format(len(pool) / ne, (len(pool) - k) / ne, -(k / ne) * 100))

    print('\n[3] 편차가 어떻게 변했나')
    print('    {:<22s}{:>12s}{:>12s}{:>12s}'.format('', '주입 전', '주입 후', '차이'))
    for 라벨, 값전, 값후 in [
            ('집단 평가 평균 편차', DEV0[idx[np.searchsorted(idx, N_TR):]].mean() * 100,
             dv[idx[np.searchsorted(idx, N_TR):]].mean() * 100),
            ('전체 평가 평균 편차', DEV0[N_TR:].mean() * 100, dv[N_TR:].mean() * 100)]:
        print('    {:<22s}{:>+12.4f}{:>+12.4f}{:>+12.4f}'.format(라벨, 값전, 값후, 값후 - 값전))
    print('    → 전체는 거의 안 움직인다 (이 집단이 전체의 {:.2%} 뿐이라서)'.format(ne / (N - N_TR)))

    print('\n[4] 점수 만들기 — 주입이 바꾸는 칸은 **평가 쪽 하나뿐**')
    a0 = _풀이(idx, DEV0, DEV0[N_TR:].mean())
    a1 = _풀이(idx, dv,   dv[N_TR:].mean())
    print('    {:<24s}{:>12s}{:>12s}   {}'.format('', '주입 전', '주입 후', '바뀌나'))
    for 라벨, 키, f in [('학습 평균 편차 (%p)', '학습평균편차', '{:>+12.4f}'),
                       ('기울기 (%p/일)',      '기울기',      '{:>+12.4f}'),
                       ('학습 평균 날짜',       '학습평균날짜', '{:>12.2f}'),
                       ('평가 평균 날짜',       '평가평균날짜', '{:>12.2f}'),
                       ('gap (일)',            'gap',        '{:>12.2f}'),
                       ('기준선 (%p)',          '기준선',      '{:>+12.4f}'),
                       ('± 기준선 오차 (%p)',   '기준선오차',   '{:>12.4f}'),
                       ('평가 평균 편차 (%p)',  '평가평균편차', '{:>+12.4f}'),
                       ('± 평가 오차 (%p)',     '평가오차',    '{:>12.4f}'),
                       ('하락 = 기준선 − 평가', '하락',        '{:>+12.4f}'),
                       ('t', 't',                             '{:>+12.2f}')]:
        같음 = abs(a0[키] - a1[키]) < 1e-9
        print(('    {:<24s}' + f + f + '   {}').format(
            라벨, a0[키], a1[키], '그대로' if 같음 else '← 바뀜'))
    print('    → 학습 쪽(평균·기울기·gap·기준선)은 **전부 그대로**. 주입을 평가 기간에만 했기 때문.')
    print('       그래서 기대 모델도 기준선 기울기도 다시 구할 필요가 없다 (계획서 7.2)')

    _g = 집단점수(idx, dv)
    assert abs(_g[2] - a1['t']) < 1e-6 and abs(_g[1] - a1['하락']) < 1e-6, '손풀이와 집단점수() 가 다름'
    print('    [대조] 집단점수() 와 일치 — 하락 {:+.4f} / t {:+.2f} (OK)'.format(_g[1], _g[2]))

    if 탐색:
        print('\n[5] 빔 서치 — 아무 말 없이 돌림 (약 40초)')
        allc = 빔서치(dv)
        top  = 추려내기(allc, 5)
        print('    {:<58s}{:>9s}{:>8s}{:>9s}'.format('상위 5개 가설', '인원', 't', '자카드'))
        순위 = 0
        for i, (r, d) in enumerate(top, 1):
            j = 자카드(rows(d), idx)
            if j >= MAX_JACC and 순위 == 0:
                순위 = i
            print('    {:<58s}{:>9,.0f}{:>+8.2f}{:>9.2f}{}'.format(
                ' AND '.join(c['이름'] for c in d)[:58], r[0], r[2], j, '  ← 정답' if j >= MAX_JACC else ''))
        원복()
        print('\n[6] 판정')
        print('    최고 t {:.2f}  vs  영분포 {:.2f}  →  {}'.format(
            allc[0][0][2], T_NULL, '넘음' if allc[0][0][2] > T_NULL else '**못 넘음 — 잡음에 묻힘**'))
        print('    정답 순위 {}  →  {}'.format(순위 or '10위 밖',
              'Hit@1' if 순위 == 1 else ('Hit@3' if 0 < 순위 <= 3 else '놓침')))
    print('=' * 78, '  ({:.0f}초)'.format(time.time() - t0))


느린걸음('channel = other', 5)

[1] 집단 — channel = other
    평가 행 28,791  /  그중 복귀한 행 22,894  (복귀율 0.7952)  /  유효 인원 3,358명
    보정 방식: 집단별 기준선

[2] 뒤집기 — δ = 5%p
    뒤집을 행 수 = 5% x 28,791 = 1,440행   (복귀한 22,894행 중 6.3%)
    집단 복귀율  0.7952  →  0.7452   (차이 -5.00%p — 정확히 δ 만큼)

[3] 편차가 어떻게 변했나
                                  주입 전        주입 후          차이
    집단 평가 평균 편차                +2.5698     -2.4317     -5.0016
    전체 평가 평균 편차                +1.4623     +1.4452     -0.0171
    → 전체는 거의 안 움직인다 (이 집단이 전체의 0.34% 뿐이라서)

[4] 점수 만들기 — 주입이 바꾸는 칸은 **평가 쪽 하나뿐**
                                    주입 전        주입 후   바뀌나
    학습 평균 편차 (%p)                +0.0337     +0.0337   그대로
    기울기 (%p/일)                   +0.1994     +0.1994   그대로
    학습 평균 날짜                        7.12        7.12   그대로
    평가 평균 날짜                       21.57       21.57   그대로
    gap (일)                        14.45       14.45   그대로
    기준선 (%p)                     +2.9142     +2.9142   그대로
    ± 기준선 오차 (%p)                 0.7492      0.7492   그